Storing数据存储组件
1、Document stores：用来存储Document和Node
2、Index stores:存储index相关的元数据
3、Vector stroes：存储聊天记录
4、Chat stores:存储；聊天记录
5、Property stores:用来存储知识图谱相关的数据

In [ ]:
from llama_index.core import VectorStoreIndex,StorageContext
from llama_index.vector_stores.redis import RedisVectorStore
from llama_index.core import SimpleDirectoryReader
from langchain_community.embeddings import DashScopeEmbeddings
from llama_index.embeddings.langchain import LangchainEmbedding
from Config.load_key import open_key
from redis import Redis

model = LangchainEmbedding(
    DashScopeEmbeddings(
        dashscope_api_key=open_key,
        model="text-embedding-v1",
    )
)

documents = SimpleDirectoryReader('./Source').load_data()

redis_client = Redis.from_url('redis://localhost:6379')
vector_store = RedisVectorStore(redis_client=redis_client, overwrite=True)

##从redis中获取数据
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(documents=documents, storage_context=storage_context, embed_model=model)

retriever = index.as_retriever()
response = retriever.retrieve("怎么退款")
for chunk in response:
    print(chunk.node.text)